# Transformer Language Model Architecture

A language model takes as input a batched sequence of integer token IDs (i.e., `torch.Tensor` of shape
(`batch_size`, `sequence_length`)), and returns a (batched) normalized probability distribution over the
vocabulary (i.e., a PyTorch Tensor of shape (`batch_size`, `sequence_length`, `vocab_size`)), where the
predicted distribution is over the next token for each input token. 

When training the language model, we use these next-token predictions to calculate the cross-entropy loss between the actual next token and the predicted next token. When generating text from the language model during inference, we take the predicted next-token distribution from the final time step (i.e., the last item in the sequence) to generate the next token in the sequence (e.g., by taking the token with the highest probability, sampling from the distribution, etc.), add the generated token to the input sequence, and repeat.

In this project, we will build this Transformer language model from scratch.

## Parameter Initialization

Pre-norm transformers are unusually robust to initializations, but they can still have a significant impact on training speed and convergence.

For now, use these approximate initializations (Normal Distribution here refer to the classic Gaussian bell curve distribution):

(a) Linear weights (e.g., in Feed Forward Neural Nets): $N(\mu = 0, \sigma^2 = \frac{2}{d_{in}+d_{out}})$, truncated at $[-3 \sigma, 3 \sigma]$, where $d_{in}$ and $d_{out}$ refer to the input and output dimensions respectively.

(b) Embedding (in the LLM context): $N(\mu = 0, \sigma^2 = 1)$, truncated at $[-3, 3]$.

(c) RMSNorm (in the LLM context): $\mathbb{1}$ -- uniformly 1's.

You should use `torch.nn.init.trunc_normal_` to initialize the truncated normal weights.


## Linear and Embedding Modules, RoPE, RMS Normalization

### Linear Module

Following most modern LLMs, we will not include a bias term.

#### Coding task for Linear Module:

Implement a `Linear` Python class that inherits from `torch.nn.Module` and performs a linear transformation. Your implementation should follow the following interface. This is intended to resemble the interface of PyTorch’s built-in `nn.Linear` module, except for not having a bias argument or parameter.

- `def __init__(self, in_features, out_features, device=None, dtype=None)` 
  - Construct a linear transformation module. This function should accept the following parameters:
    - `in_features`: `int` 
      final dimension of the input
    - `out_features`: `int` 
      final dimension of the output
    - `device`: `torch.device | None = None` 
      Device to store the parameters on
    - `dtype`: `torch.dtype | None = None` 
      Data type of the parameters

- `def forward(self, x: torch.Tensor) -> torch.Tensor` 
  - Apply the linear transformation to the input.

Make sure to:

(i) subclass `nn.Module`

(ii) call the superclass constructor

(iii) construct and store your parameter as W, putting it in an `nn.Parameter`. NOTE: $W \in \mathbb{R}^{d_{out} \times d_{in}}$ is in **row-major** form, where each row corresponds to one output feature. In `forward`, compute the transformation as $x W^T$ (equivalently, `x @ W.T`) over the final input dimension, preserving any leading dimensions of `x`.

(iv) do **not** use `nn.Linear` or `nn.functional.linear`

For initializations, use the settings from above along with `torch.nn.init.trunc_normal_` to initialize the weights.

To test your `Linear` module, implement the test adapter at [adapters.run_linear]. The adapter should load the given weights into your `Linear` module. You can use `Module.load_state_dict` for this purpose. Then, run `uv run pytest -k test_linear` and check that all unit tests pass.

### Embedding Module

Given a sequence of token IDs, the Transformer language model uses an input embedding to convert token IDs to dense vectors, passes the embedded tokens through `num_layers` Transformer blocks, and then applies a learned linear projection (the “output embedding” or “LM head”) to produce the predicted next-token logits.

The first layer of the Transformer is an embedding layer that maps integer token IDs into a vector space of dimension `d_model`. 

We will implement a custom `Embedding` class that inherits from `torch.nn.Module` (so you should not use `nn.Embedding`). The forward method should select the embedding vector for each token ID by indexing into an embedding matrix of shape (`vocab_size`, `d_model`) using a `torch.LongTensor` of token IDs with shape (`batch_size`, `sequence_length`).

#### Coding task for Embedding Module

Implement the `Embedding` class that inherits from `torch.nn.Module` and performs an embedding lookup. Your implementation should use the following interface:

- `def __init__(self, num_embeddings, embedding_dim, device=None, dtype=None)` 
  - Construct an embedding module. This function should accept the following parameters:
    - `num_embeddings`: `int` 
      Size of the vocabulary
    - `embedding_dim` : `int`
      Dimension of the embedding vectors, i.e., `d_model`
    - `device: torch.device | None = None` 
      Device to store the parameters on
    - `dtype: torch.dtype | None = None` 
      Data type of the parameters

- `def forward(self, token_ids: torch.Tensor) -> torch.Tensor` 
  - Lookup the embedding vectors for the given token IDs.

Make sure to:

(i) subclass `nn.Module`

(ii) call the superclass constructor

(iii) initialize your embedding matrix as an `nn.Parameter`

(iv) store the embedding matrix with the `d_model` being the final dimension

(v) do **not** use `nn.Embedding` or `nn.functional.embedding`

Again, use the settings from above for initialization, and use `torch.nn.init.trunc_normal_` to initialize the weights. 

To test your implementation, implement the test adapter at [adapters.run_embedding]. Then, run `uv run pytest -k test_embedding` and check that all tests pass.

### RoPE (Rotary Positional Embeddings)

To inject positional information into the model, we will implement Rotary Position Embeddings (RoPE). For a given query token $q^{(i)} = W_q x^{(i)} \in \mathbb{R}^d$ at token position $i$, we will apply a pairwise rotation matrix $R^i$, giving us $(q')^{(i)} = R^i q^{(i)} = R^i W_q x^{(i)}$. Here, $R^i$ will rotate pairs of embedding elements $q^{(i)}_{2k-1:2k}$ as $2d$-vectors by the angle $\theta_{i,k} = \frac{i}{\Theta^{(2k-2)/d}}$ for $k \in \{1, \ldots, d/2\}$ and some constant $\Theta$. Thus, we can consider $R^i$ to be a block-diagonal matrix of size $d \times d$, with blocks $R^i_k$ for $k \in \{1, \ldots, d/2\}$, with

$R^i_k =
\begin{pmatrix}
\cos(\theta_{i,k}) & -\sin(\theta_{i,k}) \\
\sin(\theta_{i,k}) & \cos(\theta_{i,k})
\end{pmatrix}$

Thus we get the full rotation matrix

$R^i =
\begin{pmatrix}
R^i_1 & 0 & 0 & \cdots & 0 \\
0 & R^i_2 & 0 & \cdots & 0 \\
0 & 0 & R^i_3 & \cdots & 0 \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
0 & 0 & 0 & \cdots & R^i_{d/2}
\end{pmatrix}$,

where the $0$'s represent $2 \times 2$ zero matrices. While one could construct the full $d \times d$ matrix, a good solution should use the properties of this matrix to implement the transformation more efficiently. Since we only care about the relative rotation of tokens within a given sequence, we can reuse the values we compute for $\cos(\theta_{i,k})$ and $\sin(\theta_{i,k})$ across layers, and different batches. If you would like to optimize it, you may use a single RoPE module referenced by all layers, and it can have a $2d$ pre-computed buffer of $\sin$ and $\cos$ values created during `init` with `self.register\_buffer(persistent=False)`, instead of an `nn.Parameter` because we do not want to learn these fixed cosine and sine values. The exact same rotation process we did for our $q^{(i)}$ is then done for $k^{(j)}$, rotating by the corresponding $R^j$. Notice that this layer has no learnable parameters.

#### Coding Task for RoPE

Implement a Python class `RotaryPositionalEmbedding` that applies RoPE to the input tensor. The following interface is to be used:

- `def __init__(self, theta: float, d_k: int, max_seq_len: int, device=None)` 
  - Construct the RoPE module and create buffers if needed.
    - `theta`: `float` 
      $\Theta$ value for the RoPE
    - `d_k`: `int` 
      dimension of query and key vectors
    - `max_seq_len`: `int` 
      Maximum sequence length that will be input
    - `device: torch.device | None = None` 
      Device to store the buffer on

- `def forward(self, x: torch.Tensor, token_positions: torch.Tensor) -> torch.Tensor` 
  - Process an input tensor of shape (..., `seq_len`, `d_k`) and return a tensor of the same shape. Note that you should tolerate $x$ with an arbitrary number of batch dimensions. You should assume that the token positions are a tensor of shape (..., `seq_len`) specifying the token positions of $x$ along the sequence dimension.

You should use the token positions to slice your (possibly precomputed) $\cos$ and $\sin$ tensors along the sequence dimension. To test your implementation, complete [adapters.run_rope] and make sure it passes `uv run pytest -k test_rope`.

### Root Mean Square Layer Normalization

We will use root mean square layer normalization.

Given a vector $a \in \mathbb{R}^{d_{model}}$ of activations, `RMSNorm` will scale each activation $a_i$ according to the formula

$\operatorname{RMSNorm}(a_i) = \frac{a_i}{\operatorname{RMS(a)}} \times g_i$,

where $\operatorname{RMS}(a) = \sqrt{ \frac{1}{d_{model}} \times ( \sum_{i=1}^{d_{model}} (a_i)^2 ) + \epsilon }$. 

Here, $g_i$ is a learnable parameter (there are `d_model` such parameters in total), and $\epsilon$ is a hyperparameter often fixed at +1e-05. 

You should upcast your input to `torch.float32` to prevent overflow when you square the input. Overall, your `forward` method should look like:

```python
in_dtype = x.dtype
x = x.to(torch.float32)
# Your code here performing RMSNorm
# ...
result = computed_result
# Return the result in the original dtype
return result.to(in_dtype)
```

#### Coding tasks for RMS Normalization

Implement `RMSNorm` as a `torch.nn.Module`. This Python class should use the following interface:

- `def __init__(self, d_model: int, eps: float = 1e-5, device=None, dtype=None)` 
  - Construct the RMSNorm module. This function should accept the following parameters:
    - `d_model`: `int` 
      Hidden dimension of the model
    - `eps: float = 1e-5` 
      Epsilon value for numerical stability
    - `device: torch.device | None = None` 
      Device to store the parameters on
    - `dtype: torch.dtype | None = None` 
      Data type of the parameters

- `def forward(self, x: torch.Tensor) -> torch.Tensor` 
  - Process an input tensor of shape (`batch_size`, `sequence_length`, `d_model`) and return a tensor of the same shape.

Note: Remember to upcast your input to `torch.float32` before performing the normalization (and later downcast to the original `dtype`), as described above.
To test your implementation, implement the test adapter at [adapters.run_rmsnorm]. Then, run `uv run pytest -k test_rmsnorm` and make sure all tests pass.

In [ ]:
from __future__ import annotations

import math

import torch


class Linear(torch.nn.Module):
    def __init__(
        self,
        in_features: int,
        out_features: int,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ) -> None:
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.W = torch.nn.Parameter(torch.empty((out_features, in_features), device=device, dtype=dtype))

        std = math.sqrt(2.0 / (in_features + out_features))
        torch.nn.init.trunc_normal_(self.W, mean=0.0, std=std, a=-3.0 * std, b=3.0 * std)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x @ self.W.T


class Embedding(torch.nn.Module):
    def __init__(
        self,
        num_embeddings: int,
        embedding_dim: int,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ) -> None:
        super().__init__()
        self.num_embeddings = num_embeddings
        self.embedding_dim = embedding_dim
        self.weight = torch.nn.Parameter(torch.empty((num_embeddings, embedding_dim), device=device, dtype=dtype))

        torch.nn.init.trunc_normal_(self.weight, mean=0.0, std=1.0, a=-3.0, b=3.0)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        return self.weight[token_ids]


class RotaryPositionalEmbedding(torch.nn.Module):
    def __init__(
        self,
        theta: float,
        d_k: int,
        max_seq_len: int,
        device: torch.device | None = None,
    ) -> None:
        super().__init__()
        if d_k % 2 != 0:
            raise ValueError("d_k must be even for RoPE pairwise rotations")

        self.theta = theta
        self.d_k = d_k
        self.max_seq_len = max_seq_len

        positions = torch.arange(max_seq_len, device=device, dtype=torch.float32)
        dimension_indices = torch.arange(0, d_k, 2, device=device, dtype=torch.float32)
        angles = positions[:, None] / (theta ** (dimension_indices / d_k))
        self.register_buffer("cos", torch.cos(angles), persistent=False)
        self.register_buffer("sin", torch.sin(angles), persistent=False)

    def forward(self, x: torch.Tensor, token_positions: torch.Tensor) -> torch.Tensor:
        in_dtype = x.dtype
        token_positions = token_positions.to(device=self.cos.device, dtype=torch.long)
        cos = self.cos[token_positions].to(device=x.device)
        sin = self.sin[token_positions].to(device=x.device)

        while cos.ndim < x.ndim:
            cos = cos.unsqueeze(-3)
            sin = sin.unsqueeze(-3)

        x_even = x[..., 0::2]
        x_odd = x[..., 1::2]
        rotated_even = x_even * cos - x_odd * sin
        rotated_odd = x_even * sin + x_odd * cos
        result = torch.stack((rotated_even, rotated_odd), dim=-1).flatten(-2)
        return result.to(in_dtype)


class RMSNorm(torch.nn.Module):
    def __init__(
        self,
        d_model: int,
        eps: float = 1e-5,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ) -> None:
        super().__init__()
        self.d_model = d_model
        self.eps = eps
        self.weight = torch.nn.Parameter(torch.ones((d_model,), device=device, dtype=dtype))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        in_dtype = x.dtype
        x = x.to(torch.float32)
        rms = torch.sqrt(torch.mean(x * x, dim=-1, keepdim=True) + self.eps)
        result = x / rms * self.weight.to(torch.float32)
        return result.to(in_dtype)


### How the `Linear`, `Embedding`, `RotaryPositionalEmbedding`, and `RMSNorm` classes work

All four classes inherit from `torch.nn.Module`, so PyTorch tracks their parameters and buffers through the module machinery. Calling `super().__init__()` sets up that machinery before any parameters or buffers are assigned. Each learnable tensor is wrapped in `torch.nn.Parameter`; assigning a `Parameter` to an attribute registers it in the module state dict, includes it in `module.parameters()`, and allows autograd to accumulate gradients into it during backpropagation.

`Linear` stores one parameter, `W`, whose shape is `(out_features, in_features)`. This is a row-major layout for linear-layer weights: each row contains the weights for one output coordinate, and each column corresponds to one input coordinate. If the input tensor `x` has shape `(..., in_features)`, the leading dimensions `...` can be a batch, a sequence, or any other collection of positions. The expression `self.W.T` views the weight matrix with shape `(in_features, out_features)`, and `x @ self.W.T` performs matrix multiplication over only the final input dimension. The result therefore has shape `(..., out_features)`: all leading dimensions are preserved, and the last dimension is replaced by the output-feature dimension. There is no bias tensor, matching the assignment specification.

The `Linear` initializer creates uninitialized storage with `torch.empty`, wraps it as `W`, then fills it in place with `torch.nn.init.trunc_normal_`. The standard deviation is `sqrt(2 / (in_features + out_features))`, so the initialization scale depends on both the fan-in and fan-out of the layer. The lower and upper truncation bounds are `-3 * std` and `3 * std`, which keeps sampled weights within three standard deviations of the zero mean.

`Embedding` stores one parameter, `weight`, whose shape is `(num_embeddings, embedding_dim)`. The first axis is a lookup table over vocabulary IDs: row `i` is the vector assigned to token ID `i`. The final axis is the dense embedding dimension `d_model`, so selecting rows naturally appends that vector dimension to the input token-ID shape. If `token_ids` has shape `(... )`, direct indexing with `self.weight[token_ids]` returns a tensor of shape `(..., embedding_dim)`. For example, a token-ID tensor of shape `(batch_size, sequence_length)` becomes embedded token vectors of shape `(batch_size, sequence_length, embedding_dim)`.

The `Embedding` initializer also uses `torch.empty` followed by `torch.nn.init.trunc_normal_`, but with mean `0`, standard deviation `1`, and fixed bounds `[-3, 3]`. The forward pass does not perform matrix multiplication. It uses tensor indexing to gather rows from the embedding table, so each integer token ID is replaced by the corresponding learned vector while the original token-ID layout is preserved as the leading dimensions of the output tensor.

`RotaryPositionalEmbedding` has no learnable parameters. During initialization, it builds the fixed RoPE angles for every cached position from `0` through `max_seq_len - 1` and for every even coordinate pair in the `d_k` query/key dimension. It stores `cos` and `sin` with `register_buffer(..., persistent=False)`, which means the tensors move with the module across devices but are not trained and are not saved as learned model weights.

The RoPE forward pass accepts `x` with shape `(..., sequence_length, d_k)` and `token_positions` with shape `(..., sequence_length)`. It indexes the cached cosine and sine buffers at those token positions, adds singleton dimensions before the sequence axis when needed so the position tensors broadcast across extra batch or head axes, and then splits the final feature dimension into even and odd coordinates. Each pair is rotated by computing `x_even * cos - x_odd * sin` for the even coordinate and `x_even * sin + x_odd * cos` for the odd coordinate. Finally, `torch.stack(..., dim=-1).flatten(-2)` interleaves the rotated pairs back into the original final dimension, and the result is returned in the input dtype.

`RMSNorm` stores one learnable scale parameter, `weight`, whose shape is `(d_model,)`. This corresponds to the vector of $g_i$ values in the RMSNorm formula, one scale value for each coordinate in the final activation dimension. The initializer uses `torch.ones` so the normalization starts as pure root-mean-square rescaling with no learned coordinate-specific change. The module also stores `d_model` and `eps` for clarity and for the normalization denominator.

The `RMSNorm` forward pass first records the input dtype and converts the activations to `torch.float32`. This keeps the square-and-average computation more stable for lower-precision inputs. It then computes `x * x`, averages over the final dimension with `keepdim=True`, adds `eps`, and takes the square root to get a denominator that broadcasts across the original input shape. Dividing `x` by this denominator normalizes each vector over its last dimension, multiplying by `weight` applies the learned per-coordinate scale, and `result.to(in_dtype)` returns the output in the same dtype as the input.


## Position-Wise Feed-Forward Network

The SiLU or Swish activation function is defined as follows:

$\operatorname{SiLU}(x) = x \times \sigma(x) = \frac{x}{1+e^{-x}}$.

Here, $\sigma(x)$ is the (logistic) sigmoid function.

The SiLU function is similar to the ReLU function (rectified linear unit) but is smooth at 0. 

Gated Linear Units (GLUs) are defined as the **element-wise product** of one linear transformation and a sigmoid-gated linear transformation:

$\operatorname{GLU}(x,W_1,W_2) = (xW_2^T) \odot \sigma(xW_1^T)$,

where $\odot$ represents element-wise multiplication, also known as the Hadamard product (to be distinguished from matrix multiplication).

Under the assumption that biases are omitted, the SwiGLU feed-forward layer, a GLU variant that uses a SiLU/Swish gate, can be expressed as follows:

$\operatorname{SwiGLU}(x,W_1,W_2,W_3) = (\operatorname{SiLU}(xW_1^T) \odot xW_3^T)W_2^T$,

where $x \in \mathbb{R}^{d_{model}}$, $W_1,W_3 \in \mathbb{R}^{d_{ff} \times d_{model}}$, $W_2 \in \mathbb{R}^{d_{model} \times d_{ff}}$, and $d_{ff} = \frac{8}{3} \times d_{model}$ (rounded in practice) -- this definition of $d_{ff}$ is common in SwiGLU Transformer feed-forward networks because it keeps the parameter count roughly comparable to a standard feed-forward network with hidden width $4 \times d_{model}$.

We shall use SwiGLU for our feed forward network:

$\operatorname{FFN}(x) = \operatorname{SwiGLU}(x,W_1,W_2,W_3)$.

### Coding task for position-wise feed-forward network

Implement the SwiGLU feed-forward network.

Note: Manually implement a numerically stable sigmoid from elementary functions. You should also set $d_{ff}$ to $\frac{8}{3} \times d_{model}$ rounded to the nearest multiple of 64 in your implementation. Accept $d_{ff}$ explicitly when provided by tests/configs, and only compute using the formula described if no $d_{ff}$ is supplied. Assume that:

- Input shape is (..., `d_model`).

- $W_1$ and $W_3$ are up-projections of shape (`d_ff`, `d_model`).

- $W_2$ is the down-projection of shape (`d_model`, `d_ff`).

To test your implementation against our provided tests, you will need to implement the test adapter at [adapters.run_swiglu]. Then, run `uv run pytest -k test_swiglu` to test your implementation.

In [ ]:
import torch

from cs336_basics.nn_linear_embedding_rope_rmsnorm import Linear


def swiglu_d_ff(d_model: int) -> int:
    raw_d_ff = 8.0 * d_model / 3.0
    return max(64, int((raw_d_ff + 32.0) // 64.0) * 64)


def stable_sigmoid(x: torch.Tensor) -> torch.Tensor:
    z = torch.exp(-torch.abs(x))
    return torch.where(x >= 0, 1.0 / (1.0 + z), z / (1.0 + z))


def silu(x: torch.Tensor) -> torch.Tensor:
    return x * stable_sigmoid(x)


class SwiGLU(torch.nn.Module):
    def __init__(
        self,
        d_model: int,
        d_ff: int | None = None,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ) -> None:
        super().__init__()
        self.d_model = d_model
        self.d_ff = d_ff if d_ff is not None else swiglu_d_ff(d_model)
        self.w1 = Linear(d_model, self.d_ff, device=device, dtype=dtype)
        self.w2 = Linear(self.d_ff, d_model, device=device, dtype=dtype)
        self.w3 = Linear(d_model, self.d_ff, device=device, dtype=dtype)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.w2(silu(self.w1(x)) * self.w3(x))


## The Multi-Head Attention and Multi-Head Self-Attention Mechanism

### Softmax

The definition of the attention operation makes use of softmax, an operation that takes an unnormalized **row vector** of scores and turns it into a normalized distribution. For $v \in \mathbb{R}^n$, define

$\operatorname{softmax}(v)_i = \frac{\exp(v_i)}{\sum_{j=1}^{n} \exp(v_j)}$.

When softmax is applied to a matrix, we will apply it row-wise unless otherwise stated.

Note that $\exp(v_i)$ can become `inf` for large values. We can avoid this by noticing that the softmax operation is invariant to adding any constant $c$ to all inputs. We can leverage this property for numerical stability. Typically, we will subtract the largest entry of $v$ from all elements of $v$, making the new largest entry $0$. For a matrix, the maximum is subtracted row-wise, not globally over the whole matrix.

### Scaled Dot-Product Attention

Define the attention operation as follows:

$\operatorname{Attention}(Q,K,V)=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$,

where $Q \in \mathbb{R}^{n \times d_k}$, $K \in \mathbb{R}^{m \times d_k}$, and $V \in \mathbb{R}^{m \times d_v}$. Here, $Q$, $K$, and $V$ are the query, key, and value activations supplied to the attention operation. They are not themselves learnable parameters.

The matrix $QK^T$ has shape $n \times m$. Its $(i,j)$-entry measures the raw dot-product compatibility (before scaling) between query $i$ and key $j$. The softmax is applied **row-wise**, so each query position obtains a probability distribution over the $m$ key positions. The final output has shape $n \times d_v$.

It is sometimes necessary to mask the attention scores before applying the softmax. We use a boolean mask $M \in \{\mathrm{True},\mathrm{False}\}^{n \times m}$, where $M_{ij}=\mathrm{True}$ means that query $i$ is allowed to attend to key $j$, and $M_{ij}=\mathrm{False}$ means that query $i$ is not allowed to attend to key $j$.

Let $S=QK^T/\sqrt{d_k}$. The masked score matrix $S'$ is defined by setting $S'_{ij}=S_{ij}$ when $M_{ij}=\mathrm{True}$, and $S'_{ij}=-\infty$ when $M_{ij}=\mathrm{False}$. The masked attention operation is then

$\operatorname{Attention}(Q,K,V;M)=\operatorname{softmax}(S')V$.

For causal attention, a single masked $n \times n$ attention computation computes all positions in parallel, instead of separately computing attention over  the prefix $1,\ldots,i$ for each query position $i$.

We assume each row of the mask has at least one $\mathrm{True}$ entry. If an entire row is masked out, then the corresponding row of $S'$ is all $-\infty$, and the softmax is undefined numerically.

### Multi-Head Attention and Multi-Head Self-Attention

For multi-head attention, suppose

$Q \in \mathbb{R}^{n \times h d_k}$, $K \in \mathbb{R}^{m \times h d_k}$, and $V \in \mathbb{R}^{m \times h d_v}$.

We split these matrices along the feature dimension into $h$ heads:

$Q_i \in \mathbb{R}^{n \times d_k}$, $K_i \in \mathbb{R}^{m \times d_k}$, and $V_i \in \mathbb{R}^{m \times d_v}$,

for $i=1,\ldots,h$. Then

$\operatorname{head}_i=\operatorname{Attention}(Q_i,K_i,V_i)$

has shape $n \times d_v$, and

$\operatorname{MultiHead}(Q,K,V)=\operatorname{Concat}(\operatorname{head}_1,\ldots,\operatorname{head}_h)$

has shape $n \times h d_v$, where concatenation is along the feature dimension. Note that $\operatorname{MultiHead}(Q,K,V)$ denotes only the pre-output-projection concatenation of the individual attention heads; the final output projection is included separately in $\operatorname{MultiHeadSelfAttention}$, described as follows.

For self-attention, let $x \in \mathbb{R}^{n \times d_{model}}$. We compute

$\operatorname{MultiHeadSelfAttention}(x)=\operatorname{MultiHead}(xW_Q^T,xW_K^T,xW_V^T)W_O^T$,

where the learnable parameters are

$W_Q \in \mathbb{R}^{h d_k \times d_{model}}$,

$W_K \in \mathbb{R}^{h d_k \times d_{model}}$,

$W_V \in \mathbb{R}^{h d_v \times d_{model}}$,

and

$W_O \in \mathbb{R}^{d_{model} \times h d_v}$.

Thus $xW_Q^T$ and $xW_K^T$ have shape $n \times h d_k$, while $xW_V^T$ has shape $n \times h d_v$. These are then split into $h$ heads.

Here, we use the PyTorch-style convention in which a linear layer with input dimension $a$ and output dimension $b$ has weight matrix in $\mathbb{R}^{b \times a}$ and computes $xW^T$. For simplicity, we also defer discussion of the batch dimension until later. Also, in many transformer implementations, especially GPT-style models, one may take $d_k=d_v=d_{head}$ and $h d_{head}=d_{model}$. The more general notation above does not require this equality.

### Causal Masking

In causal self-attention, we have $m=n$, and we use a causal mask satisfying $M_{ij}=\mathrm{True}$ if and only if $j \leq i$. Thus each position may attend only to itself and earlier positions.

Using the convention that `True` means “allowed to attend”, we can construct this mask in PyTorch by

```python
M = ~torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)
```

In an implementation where the input tensor $x$ is on a specific device, one should construct the mask on the same device:

```python
M = ~torch.triu(
    torch.ones(n, n, dtype=torch.bool, device=x.device),
    diagonal=1,
)
```

When a mask $M$ is used, define

$\operatorname{MultiHead}(Q,K,V;M) = \operatorname{Concat}(\operatorname{head}_1,\ldots,\operatorname{head}_h)$,

where $\operatorname{head}_i=\operatorname{Attention}(Q_i,K_i,V_i;M)$ for $i=1,\ldots,h$. 

Therefore causal self-attention is

$\operatorname{MultiHeadSelfAttention}(x;M) = \operatorname{MultiHead}(xW_Q^T,xW_K^T,xW_V^T;M)W_O^T$.

### Batching in Scaled Dot-Product Attention

The discussion above omits batch dimensions for simplicity. In an implementation, however, attention is usually applied to many independent sequences at once. More generally, we may allow any number of leading batch-like dimensions.

Suppose

$Q \in \mathbb{R}^{B_1 \times \cdots \times B_r \times n \times d_k}$,

$K \in \mathbb{R}^{B_1 \times \cdots \times B_r \times m \times d_k}$,

and

$V \in \mathbb{R}^{B_1 \times \cdots \times B_r \times m \times d_v}$.

The leading dimensions $B_1,\ldots,B_r$ are batch-like dimensions. Attention is applied independently for each fixed batch index. In other words, for each $(b_1,\ldots,b_r)$, we compute

$\operatorname{Attention}(Q_{b_1,\ldots,b_r},K_{b_1,\ldots,b_r},V_{b_1,\ldots,b_r})$,

where

$Q_{b_1,\ldots,b_r} \in \mathbb{R}^{n \times d_k}$,

$K_{b_1,\ldots,b_r} \in \mathbb{R}^{m \times d_k}$,

and

$V_{b_1,\ldots,b_r} \in \mathbb{R}^{m \times d_v}$.

No attention is computed across different batch entries. The batch dimensions merely allow many independent attention computations to be carried out in parallel.

In the batched setting, the score tensor is computed by a batched matrix multiplication over the final two dimensions:

$S = QK^{T_{\mathrm{last-two}}}/\sqrt{d_k}$,

where $K^{T_{\mathrm{last-two}}}$ denotes transposition of the final two dimensions. Thus

$S \in \mathbb{R}^{B_1 \times \cdots \times B_r \times n \times m}$.

In PyTorch, this corresponds to using

```python
scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)
```

rather than applying a global two-dimensional transpose to the entire tensor.

The softmax is applied along the final dimension of $S$, namely the key-position dimension. Thus, for each fixed batch index and query position, the attention weights over the $m$ key positions sum to $1$.

If a boolean mask $M \in \{\mathrm{True},\mathrm{False}\}^{n \times m}$ is provided, then $M$ is broadcast over all leading batch-like dimensions. Thus the same mask is applied independently to every batch entry. With the convention that $\mathrm{True}$ means “allowed to attend”, we define the masked score tensor $S'$ by setting $(S')_{b_1,\ldots,b_r,i,j} = S_{b_1,\ldots,b_r,i,j}$ if $M_{ij}=\mathrm{True}$, and $-\infty$ if $M_{ij}=\mathrm{False}$.

Equivalently, in PyTorch one may write

```python
scores = scores.masked_fill(~mask, float("-inf"))
```

where `scores` has shape `(..., n, m)` and `mask` has shape `(n, m)`. The mask broadcasts over the leading dimensions of `scores`.

The masked attention output is then

$\operatorname{Attention}(Q,K,V;M)=\operatorname{softmax}(S')V$,

where the final multiplication is again a batched matrix multiplication over the final two dimensions. The output has shape

$B_1 \times \cdots \times B_r \times n \times d_v$.

We assume that every row of the mask has at least one $\mathrm{True}$ entry. If an entire row is masked out, then the corresponding row of $S'$ consists entirely of $-\infty$, and the softmax is numerically undefined.

### RoPE in Batched Multi-Head Attention

Rotary positional embedding, or RoPE, should be applied to the query and key vectors before computing the attention scores. It should not be applied to the value vectors.

After the query, key, and value projections have been split into heads, suppose

$Q \in \mathbb{R}^{B \times h \times n \times d_k}$,

$K \in \mathbb{R}^{B \times h \times m \times d_k}$,

and

$V \in \mathbb{R}^{B \times h \times m \times d_v}$,

where $B$ is the batch size, $h$ is the number of heads, $n$ is the number of query positions, and $m$ is the number of key-value positions. In causal self-attention, we usually have $m=n$. RoPE is applied position-wise to $Q$ and $K$. Treating each last-dimensional slice as a row vector in $\mathbb{R}^{1 \times d_k}$, define

$\widetilde{Q}_{b,a,t,:}=Q_{b,a,t,:}R_t^T$

and

$\widetilde{K}_{b,a,s,:}=K_{b,a,s,:}R_s^T$,

where $R_t,R_s \in \mathbb{R}^{d_k \times d_k}$ are the RoPE rotation matrices for the corresponding sequence positions. The value tensor is left unchanged:

$\widetilde{V}_{b,a,s,:}=V_{b,a,s,:}$.

The masked attention operation is then computed as

$\operatorname{Attention}(\widetilde{Q},\widetilde{K},\widetilde{V};M)$.

Here $\operatorname{Attention}$ is understood in the batched sense: the leading dimensions $B$ and $h$ are treated as batch-like dimensions, and attention is applied independently for each batch entry and each head. Here also, the indices $t$ and $s$ should be understood as sequence positions, not merely local tensor indices. In cached decoding, the RoPE position used for a query or key should be its absolute position in the generated sequence.

The RoPE rotation itself is usually shared across heads. That is, for a fixed sequence position, the same rotation is applied to the query or key vector in every head. In implementation terms, the RoPE angles should broadcast over the batch dimension and the head dimension, but should vary over the sequence-position dimension and the feature-pair dimension inside $d_k$.

RoPE acts on pairs of coordinates in the head dimension, so the rotated part of the head dimension should be even. In the simplest case, RoPE is applied to all of $d_k$, in which case $d_k$ itself should be even. Some implementations apply RoPE only to a smaller even-dimensional subspace of the head dimension.

Thus the usual order of operations is:

```python
q = apply_rope(q)
k = apply_rope(k)
out = scaled_dot_product_attention(q, k, v, mask)
```

and this should be avoided:

```python
v = apply_rope(v)
```

The reason is that RoPE modifies the geometry of query-key comparisons, and hence affects the attention scores $QK^T$. The value vectors are the content vectors that are averaged using the resulting attention probabilities, so they are not position-rotated by RoPE.

### Coding tasks

#### SoftMax function

Write a function to apply the softmax operation on a tensor. Your function should take two parameters: a tensor and a dimension `𝑖`, and apply softmax to the `𝑖`-th dimension of the input tensor. The output tensor should have the same shape as the input tensor, but its `𝑖`-th dimension will now have a normalized probability distribution. Use the trick of subtracting the maximum value in the `𝑖`-th dimension from all elements of the `𝑖`-th dimension to avoid numerical stability issues.

Test your implementation by completing [adapters.run_softmax] and making sure it passes `uv run pytest -k test_softmax_matches_pytorch`.

#### Scaled Dot Product Attention with Batching

Implement the scaled dot-product attention function. Your implementation should handle keys and queries of shape (`batch_size`, ..., `seq_len`, `d_k`) and values of shape (`batch_size`, ..., `seq_len`, `d_v`), where ... represents any number of other batch-like dimensions (if provided). The implementation should return an output with the shape (`batch_size`, ..., `seq_len`, `d_v`). 

Your implementation should also support an optional user-provided boolean mask of shape (`seq_len`, `seq_len`). The attention probabilities of positions with a mask value of True should collectively sum to 1, and the attention probabilities of positions with a mask value of False should be zero.

To test your implementation against our provided tests, you will need to implement the test adapter at [adapters.run_scaled_dot_product_attention]. The command `uv run pytest -k test_scaled_dot_product_attention` tests your implementation on third-order input tensors, while `uv run pytest -k test_4d_scaled_dot_product_attention` tests your implementation on fourth-order input tensors.

#### Causal multi-head self-attention

Implement causal multi-head self-attention as a `torch.nn.Module`. Your implementation should accept (at least) the following parameters:

- `d_model`: `int` 
    Dimensionality of the Transformer block inputs.
- `num_heads`: `int` 
    Number of heads to use in multi-head self-attention.

We will set $𝑑_𝑘 = 𝑑_𝑣 = h \times 𝑑_{model}$. Caution: keep this as the sentence fragment for the regular implementation discussion elsewhere in the assignment. For the test adapter for this specific function, use the standard GPT-style per-head dimension also discussed above: `head_dim = d_model // num_heads`, so the query, key, and value projections each have output width `d_model` and are reshaped into `num_heads` heads.

To test your implementation against our provided tests, implement the test adapter at [adapters.run_multihead_self_attention] . Then, run `uv run pytest -k test_multihead_self_attention` to test your implementation.

In [ ]:
from __future__ import annotations

import math

import torch

from cs336_basics.nn_linear_embedding_rope_rmsnorm import Linear, RotaryPositionalEmbedding


def softmax(x: torch.Tensor, dim: int) -> torch.Tensor:
    shifted = x - torch.max(x, dim=dim, keepdim=True).values
    exp_shifted = torch.exp(shifted)
    return exp_shifted / torch.sum(exp_shifted, dim=dim, keepdim=True)


def scaled_dot_product_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    mask: torch.Tensor | None = None,
) -> torch.Tensor:
    d_k = q.shape[-1]
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)

    if mask is not None:
        mask = mask.to(device=scores.device, dtype=torch.bool)
        scores = scores.masked_fill(~mask, float("-inf"))

    attention_weights = softmax(scores, dim=-1)
    return attention_weights @ v


class CausalMultiHeadSelfAttention(torch.nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        max_seq_len: int | None = None,
        theta: float | None = None,
        use_rope: bool = False,
        device: torch.device | None = None,
        dtype: torch.dtype | None = None,
    ) -> None:
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.q_proj = Linear(d_model, d_model, device=device, dtype=dtype)
        self.k_proj = Linear(d_model, d_model, device=device, dtype=dtype)
        self.v_proj = Linear(d_model, d_model, device=device, dtype=dtype)
        self.output_proj = Linear(d_model, d_model, device=device, dtype=dtype)

        if use_rope:
            if max_seq_len is None or theta is None:
                raise ValueError("max_seq_len and theta are required when use_rope=True")
            self.rope = RotaryPositionalEmbedding(theta, self.head_dim, max_seq_len, device=device)
        else:
            self.rope = None

    def _project_to_heads(self, x: torch.Tensor, projection: Linear) -> torch.Tensor:
        projected = projection(x)
        projected = projected.reshape(*projected.shape[:-1], self.num_heads, self.head_dim)
        return projected.transpose(-3, -2)

    def forward(self, x: torch.Tensor, token_positions: torch.Tensor | None = None) -> torch.Tensor:
        sequence_length = x.shape[-2]

        q = self._project_to_heads(x, self.q_proj)
        k = self._project_to_heads(x, self.k_proj)
        v = self._project_to_heads(x, self.v_proj)

        if self.rope is not None:
            if token_positions is None:
                token_positions = torch.arange(sequence_length, device=x.device)
            q = self.rope(q, token_positions)
            k = self.rope(k, token_positions)

        causal_mask = torch.tril(
            torch.ones((sequence_length, sequence_length), device=x.device, dtype=torch.bool)
        )
        attention_output = scaled_dot_product_attention(q, k, v, causal_mask)
        attention_output = attention_output.transpose(-3, -2).reshape(*x.shape[:-1], self.d_model)
        return self.output_proj(attention_output)


### How the `softmax`, scaled dot-product attention, and causal multi-head self-attention code works

The implementation keeps the assignment-required attention logic in `cs336_basics/nn_attention.py`. The `softmax` helper manually normalizes along the requested dimension by subtracting the per-dimension maximum, exponentiating the shifted values, and dividing by the per-dimension sum. This avoids the overflow problem described above without using PyTorch's built-in softmax helpers.

`scaled_dot_product_attention` treats every leading dimension as batch-like. It multiplies each query matrix by the corresponding transposed key matrix over the final two dimensions, scales the scores by `sqrt(d_k)`, applies an optional boolean mask with `True` meaning that a key position is visible, and then multiplies the resulting attention probabilities by the value tensor. Because the softmax is taken over the final dimension, each query position gets an independent distribution over key positions.

`CausalMultiHeadSelfAttention` composes the previously implemented custom `Linear` module for the query, key, value, and output projections. For the test adapter in this task, it follows the standard per-head sizing from the theory discussion, `head_dim = d_model // num_heads`, even though the assignment text above preserves the fragment $d_k = d_v = h \times d_{model}$ for the broader regular-implementation discussion. Each projected tensor is reshaped from `(..., sequence_length, d_model)` into `(..., num_heads, sequence_length, head_dim)`, so attention is computed independently for each head while preserving arbitrary leading batch dimensions. When RoPE is enabled, the shared `RotaryPositionalEmbedding` module rotates only the query and key heads before attention; value heads are left unchanged. A lower-triangular causal mask ensures each position attends only to itself and earlier positions, after which the head outputs are transposed back, flattened to `d_model`, and passed through the output projection.
